In [1]:
!git clone https://github.com/jaywalnut310/vits.git

Cloning into 'vits'...
remote: Enumerating objects: 81, done.
remote: Total 81 (delta 0), reused 0 (delta 0), pack-reused 81 (from 1)
Receiving objects: 100% (81/81), 3.33 MiB | 15.17 MiB/s, done.
Resolving deltas: 100% (22/22), done.


In [2]:
%cd vits

/Users/diyagoel/slp-diss/SOTA_models_experiments/vits


In [4]:
# NOTE: the original requirements.txt pins 2021-era versions (numpy 1.18.5, torch 1.6.0, ...)
# that have no arm64 macOS wheels and won't build on Python 3.11. Install modern equivalents
# instead. numpy is held <2 because the 2021 VITS code uses APIs removed in numpy 2.x.
!pip install "numpy<2" Cython librosa matplotlib scipy tensorboard phonemizer Unidecode torch torchvision

  Using cached cython-3.2.5-cp311-cp311-macosx_11_0_arm64.whl.metadata (7.1 kB)
  Using cached librosa-0.11.0-py3-none-any.whl.metadata (8.7 kB)
  Using cached matplotlib-3.10.9-cp311-cp311-macosx_11_0_arm64.whl.metadata (52 kB)
  Using cached scipy-1.17.1-cp311-cp311-macosx_14_0_arm64.whl.metadata (62 kB)
  Using cached Unidecode-1.4.0-py3-none-any.whl.metadata (13 kB)
  Using cached torch-2.12.0-cp311-cp311-macosx_14_0_arm64.whl.metadata (31 kB)
  Using cached audioread-3.1.0-py3-none-any.whl.metadata (9.0 kB)
  Using cached numba-0.65.1-cp311-cp311-macosx_12_0_arm64.whl.metadata (2.9 kB)
  Using cached scikit_learn-1.9.0-cp311-cp311-macosx_12_0_arm64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached soundfile-0.14.0-py2.py3-none-macosx_11_0_arm64.whl.metadata (18 kB)
  Using cached pooch-1.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached soxr-1.1.0-cp311-cp311-macosx_11_0_arm64.whl.metadata (5.8 kB)
  Using cached lazy_loader-0.5-

In [5]:
# Build the Cython monotonic_align extension. Run from the vits/ dir (cell-1 already cd'd there).
%cd monotonic_align

/Users/diyagoel/slp-diss/SOTA_models_experiments/vits/monotonic_align


In [ ]:
# __init__.py does `from .monotonic_align.core import maximum_path_c`, so the compiled .so
# must live in a NESTED monotonic_align/monotonic_align/ folder. Modern setuptools (unlike
# the 2021 Colab one) won't auto-create it, so make it first. -p is a safe no-op if it exists.
!mkdir -p monotonic_align
!python setup.py build_ext --inplace

In [18]:
!brew unlink espeak-ng
!brew install espeak

Unlinking /opt/homebrew/Cellar/espeak-ng/1.52.0... 12 symlinks removed.
==> Would install 1 formula:
espeak
==> Downloading https://ghcr.io/v2/homebrew/core/espeak/manifests/1.48.04_1-1
Already downloaded: /Users/diyagoel/Library/Caches/Homebrew/downloads/4c56bf1b34893cd2cc1cd9436bac374dbdd7803ae3a6f08b552ad51f47e29ec6--espeak-1.48.04_1-1.bottle_manifest.json
==> Fetching downloads for: espeak
Replacement:
  brew install --formula espeak-ng
⠋ Bottle espeak (1.48.04_1) ######################## Downloading   1.6MB/  1.6MB✔︎ Bottle espeak (1.48.04_1)                          Downloaded    1.6MB/  1.6MB
==> Pouring espeak--1.48.04_1.arm64_sequoia.bottle.1.tar.gz
🍺  /opt/homebrew/Cellar/espeak/1.48.04_1: 299 files, 3.2MB
==> Running `brew cleanup espeak`...
Disable this behaviour by setting `HOMEBREW_NO_INSTALL_CLEANUP=1`.
Hide these hints with `HOMEBREW_NO_ENV_HINTS=1` (see `man brew`).
==> `brew cleanup` has not been run in the last 30 days, running now...
Disable this behaviour by settin

In [9]:
%cd ..

/Users/diyagoel/slp-diss/SOTA_models_experiments/vits


In [10]:
!pip install gdown

  Using cached tqdm-4.68.2-py3-none-any.whl.metadata (58 kB)
Using cached tqdm-4.68.2-py3-none-any.whl (78 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [gdown]


In [20]:
!gdown 'https://drive.google.com/uc?id=1q86w74Ygw2hNzYP9cWkeClGT5X25PvBT'

Downloading...
From (original): https://drive.google.com/uc?id=1q86w74Ygw2hNzYP9cWkeClGT5X25PvBT
From (redirected): https://drive.google.com/uc?id=1q86w74Ygw2hNzYP9cWkeClGT5X25PvBT&confirm=t&uuid=639ede57-b4c9-44ea-9e21-5bd981123db6
To: /Users/diyagoel/slp-diss/SOTA_models_experiments/vits/pretrained_ljs.pth
100%|███████████████████████████████████████| 146M/146M [00:10<00:00, 13.5MB/s]


In [21]:
# phonemizer can't auto-locate Homebrew's espeak library on Apple Silicon
# (ctypes.util.find_library doesn't search /opt/homebrew/lib), so point it there explicitly.
import os
os.environ['PHONEMIZER_ESPEAK_LIBRARY'] = '/opt/homebrew/lib/libespeak.dylib'

%matplotlib inline
import matplotlib.pyplot as plt
import IPython.display as ipd

import json
import math
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader

import commons
import utils
from data_utils import TextAudioLoader, TextAudioCollate, TextAudioSpeakerLoader, TextAudioSpeakerCollate
from models import SynthesizerTrn
from text.symbols import symbols
from text import text_to_sequence

from scipy.io.wavfile import write


def get_text(text, hps):
    text_norm = text_to_sequence(text, hps.data.text_cleaners)
    if hps.data.add_blank:
        text_norm = commons.intersperse(text_norm, 0)
    text_norm = torch.LongTensor(text_norm)
    return text_norm

DEBUG:matplotlib.pyplot:Loaded backend inline version unknown.


In [14]:
hps = utils.get_hparams_from_file("./configs/ljs_base.json")

In [15]:
net_g = SynthesizerTrn(
    len(symbols),
    hps.data.filter_length // 2 + 1,
    hps.train.segment_size // hps.data.hop_length,
    **hps.model)
_ = net_g.eval()

_ = utils.load_checkpoint("pretrained_ljs.pth", net_g, None)

/Users/diyagoel/miniconda3/envs/vits/lib/python3.11/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


INFO:root:Loaded checkpoint 'pretrained_ljs.pth' (iteration 0)


In [22]:
stn_tst = get_text("We propose VITS, Conditional Variational Autoencoder with Adversarial Learning for End-to-End Text-to-Speech.", hps)
with torch.no_grad():
    x_tst = stn_tst.unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)])
    audio = net_g.infer(x_tst, x_tst_lengths, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.float().numpy()
ipd.display(ipd.Audio(audio, rate=hps.data.sampling_rate))

## Multi Speaker

In [23]:
!gdown 'https://drive.google.com/uc?id=11aHOlhnxzjpdWDpsz1vFDCzbeEfoIxru'

Downloading...
From (original): https://drive.google.com/uc?id=11aHOlhnxzjpdWDpsz1vFDCzbeEfoIxru
From (redirected): https://drive.google.com/uc?id=11aHOlhnxzjpdWDpsz1vFDCzbeEfoIxru&confirm=t&uuid=c1a9b7f6-464d-40d8-90fc-00fd780195fe
To: /Users/diyagoel/slp-diss/SOTA_models_experiments/vits/pretrained_vctk.pth
100%|███████████████████████████████████████| 159M/159M [00:08<00:00, 17.7MB/s]


In [24]:
hps_ms = utils.get_hparams_from_file("./configs/vctk_base.json")

In [25]:
net_g_ms = SynthesizerTrn(
    len(symbols),
    hps_ms.data.filter_length // 2 + 1,
    hps_ms.train.segment_size // hps.data.hop_length,
    n_speakers=hps_ms.data.n_speakers,
    **hps_ms.model)
_ = net_g.eval()

_ = utils.load_checkpoint("pretrained_vctk.pth", net_g_ms, None)

INFO:root:Loaded checkpoint 'pretrained_vctk.pth' (iteration 0)


In [26]:
sid = torch.LongTensor([4]) # speaker identity
stn_tst = get_text("We propose VITS, Conditional Variational Autoencoder with Adversarial Learning for End-to-End Text-to-Speech.", hps_ms)

with torch.no_grad():
    x_tst = stn_tst.unsqueeze(0)
    x_tst_lengths = torch.LongTensor([stn_tst.size(0)])
    audio = net_g_ms.infer(x_tst, x_tst_lengths, sid=sid, noise_scale=.667, noise_scale_w=0.8, length_scale=1)[0][0,0].data.float().numpy()
ipd.display(ipd.Audio(audio, rate=hps_ms.data.sampling_rate))

## Voice Conversion

In [ ]:
!wget https://jaywalnut310.github.io/vits-demo/wavs/vc_01_01.wav

In [ ]:
from mel_processing import spectrogram_torch
from utils import load_wav_to_torch

audio, sampling_rate = load_wav_to_torch("./vc_01_01.wav")

y = audio / hps_ms.data.max_wav_value
y = y.unsqueeze(0)

spec = spectrogram_torch(y, hps_ms.data.filter_length,
    hps_ms.data.sampling_rate, hps_ms.data.hop_length, hps_ms.data.win_length,
    center=False)
spec_lengths = torch.LongTensor([spec.size(-1)])
sid_src = torch.LongTensor([81])

In [ ]:
with torch.no_grad():
    sid_tgt1 = torch.LongTensor([77])
    sid_tgt2 = torch.LongTensor([14])
    sid_tgt3 = torch.LongTensor([1])
    sid_tgt4 = torch.LongTensor([17])
    audio1 = net_g_ms.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt1)[0][0,0].data.float().numpy()
    audio2 = net_g_ms.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt2)[0][0,0].data.float().numpy()
    audio3 = net_g_ms.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt3)[0][0,0].data.float().numpy()
    audio4 = net_g_ms.voice_conversion(spec, spec_lengths, sid_src=sid_src, sid_tgt=sid_tgt4)[0][0,0].data.float().numpy()
print("Original SID: %d" % sid_src.item())
ipd.display(ipd.Audio(y[0].numpy(), rate=hps_ms.data.sampling_rate))
print("Converted SID: %d" % sid_tgt1.item())
ipd.display(ipd.Audio(audio1, rate=hps_ms.data.sampling_rate))
print("Converted SID: %d" % sid_tgt2.item())
ipd.display(ipd.Audio(audio2, rate=hps_ms.data.sampling_rate))
print("Converted SID: %d" % sid_tgt3.item())
ipd.display(ipd.Audio(audio3, rate=hps_ms.data.sampling_rate))
print("Converted SID: %d" % sid_tgt4.item())
ipd.display(ipd.Audio(audio4, rate=hps_ms.data.sampling_rate))

Original SID: 81


Converted SID: 77


Converted SID: 14


Converted SID: 1


Converted SID: 17
